In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def custom_kfold_splits(n_samples: int, k_folds: int, shuffle: bool = True, random_state: int = 42):
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(random_state)
        indices = rng.permutation(indices)

    fold_sizes = np.full(k_folds, n_samples // k_folds, dtype=int)
    fold_sizes[: n_samples % k_folds] += 1

    splits = []
    start = 0
    for size in fold_sizes:
        stop = start + size
        val_idx = indices[start:stop]
        train_idx = np.concatenate([indices[:start], indices[stop:]])
        splits.append((train_idx, val_idx))
        start = stop
    return splits


def custom_stratified_kfold_splits(y_arr: np.ndarray, k_folds: int, shuffle: bool = True, random_state: int = 42):
    n_samples = len(y_arr)
    rng = np.random.default_rng(random_state)
    classes, y_encoded = np.unique(y_arr, return_inverse=True)
    fold_buckets = [[] for _ in range(k_folds)]

    for c_idx in range(len(classes)):
        c_indices = np.flatnonzero(y_encoded == c_idx)
        if shuffle:
            c_indices = rng.permutation(c_indices)
        for pos, sample_idx in enumerate(c_indices):
            fold_buckets[pos % k_folds].append(sample_idx)

    all_indices = np.arange(n_samples)
    splits = []
    for f in range(k_folds):
        val_idx = np.sort(np.array(fold_buckets[f], dtype=int))
        train_mask = np.ones(n_samples, dtype=bool)
        train_mask[val_idx] = False
        splits.append((all_indices[train_mask], val_idx))
    return splits